# Pipeline 1 – StatsBomb: Bronze ➜ Silver (incremental)

**Input (bronze):** `s3://<bucket>/bronze/statsbomb/{matches,lineups,events}/competition_id=*/season_id=*/`

**Output (silver, Parquet):**

| table | grain | partitioned by |
|---|---|---|
| `matches` | 1 row / match | competition_id, season_id |
| `events` | 1 row / event (flattened) | competition_id, season_id |
| `player_match_minutes` | 1 row / player / match | competition_id, season_id |
| `player_season_stats` | 1 row / player / competition-season (counts + per-90) | competition_id, season_id |

**Incremental logic:** only matches that are not yet in `silver/statsbomb/matches` are read from bronze.
New rows are *appended* to the match-level tables, and `player_season_stats` is recomputed **only for the
competition-seasons touched by the new matches** (dynamic partition overwrite).

In [ ]:
%idle_timeout 30
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2

In [ ]:
# --- Glue boilerplate: Spark + Glue contexts and job parameters ---
import sys
import time
import json
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

# Job parameters are passed by the Workflow trigger (--BUCKET ...); interactive runs fall back to defaults
try:
    args = getResolvedOptions(sys.argv, ["JOB_NAME", "BUCKET"])
except Exception:
    args = {"JOB_NAME": "pl_01_statsbomb_bronze_to_silver", "BUCKET": "football-analytics-moneyball-<ACCOUNT_ID>"}
job.init(args["JOB_NAME"], args)

In [ ]:
# --- Configuration ---
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

BRONZE = f"s3://{args['BUCKET']}/bronze/statsbomb"
SILVER = f"s3://{args['BUCKET']}/silver/statsbomb"
PIPELINE = "pl_01_statsbomb"

# Overwrite only the partitions present in the DataFrame being written (needed for incremental recompute)
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
RUN_START = time.time()

In [ ]:
# --- Helpers: storage I/O + observability ---
def list_paths(glob_pattern):
    """List files matching a glob using the Hadoop FS API (works for s3:// and local paths)."""
    path = spark._jvm.org.apache.hadoop.fs.Path(glob_pattern)
    fs = path.getFileSystem(spark._jsc.hadoopConfiguration())
    statuses = fs.globStatus(path) or []
    return [s.getPath().toString() for s in statuses]


def table_exists(path):
    """A table exists once at least one Parquet part file has been written under it."""
    return len(list_paths(f"{path}/*/*/*.parquet")) > 0


def write_table(df, path, partition_cols, mode):
    """Write a Parquet table (snappy) partitioned by the given columns."""
    df.write.mode(mode).partitionBy(*partition_cols).parquet(path)


def emit_metrics(metrics):
    """Structured log line (lands in CloudWatch Logs) + custom CloudWatch metrics."""
    print(json.dumps({"pipeline": PIPELINE, **metrics}))
    try:
        import boto3
        boto3.client("cloudwatch").put_metric_data(
            Namespace="FootballAnalytics/Pipelines",
            MetricData=[
                {"MetricName": k, "Value": float(v), "Unit": "Count",
                 "Dimensions": [{"Name": "Pipeline", "Value": PIPELINE}]}
                for k, v in metrics.items() if isinstance(v, (int, float))
            ],
        )
    except Exception as exc:  # metrics must never fail the pipeline
        print(f"CloudWatch metrics not published: {exc}")

## 1. Explicit schemas (schema-on-read contract)
Event files are deeply nested and heterogeneous – a small incremental batch may not contain every
event type, so relying on schema inference would make columns appear/disappear between runs.
We declare only the fields we use; Spark ignores everything else in the JSON.

In [ ]:
named = T.StructType([T.StructField("id", T.LongType()), T.StructField("name", T.StringType())])
coords = T.ArrayType(T.DoubleType())

event_schema = T.StructType([
    T.StructField("id", T.StringType()),
    T.StructField("index", T.LongType()),
    T.StructField("period", T.IntegerType()),
    T.StructField("minute", T.IntegerType()),
    T.StructField("second", T.IntegerType()),
    T.StructField("type", named),
    T.StructField("possession", T.IntegerType()),
    T.StructField("possession_team", named),
    T.StructField("play_pattern", named),
    T.StructField("team", named),
    T.StructField("player", named),
    T.StructField("position", named),
    T.StructField("location", coords),
    T.StructField("under_pressure", T.BooleanType()),
    T.StructField("pass", T.StructType([
        T.StructField("recipient", named),
        T.StructField("length", T.DoubleType()),
        T.StructField("end_location", coords),
        T.StructField("height", named),
        T.StructField("type", named),
        T.StructField("outcome", named),
        T.StructField("cross", T.BooleanType()),
        T.StructField("shot_assist", T.BooleanType()),
        T.StructField("goal_assist", T.BooleanType()),
    ])),
    T.StructField("carry", T.StructType([T.StructField("end_location", coords)])),
    T.StructField("shot", T.StructType([
        T.StructField("statsbomb_xg", T.DoubleType()),
        T.StructField("end_location", coords),
        T.StructField("key_pass_id", T.StringType()),
        T.StructField("type", named),
        T.StructField("outcome", named),
        T.StructField("body_part", named),
    ])),
    T.StructField("dribble", T.StructType([T.StructField("outcome", named)])),
    T.StructField("duel", T.StructType([T.StructField("type", named), T.StructField("outcome", named)])),
    T.StructField("interception", T.StructType([T.StructField("outcome", named)])),
    T.StructField("ball_recovery", T.StructType([T.StructField("recovery_failure", T.BooleanType())])),
])

match_schema = T.StructType([
    T.StructField("match_id", T.LongType()),
    T.StructField("match_date", T.StringType()),
    T.StructField("competition", T.StructType([
        T.StructField("competition_id", T.LongType()), T.StructField("competition_name", T.StringType())])),
    T.StructField("season", T.StructType([
        T.StructField("season_id", T.LongType()), T.StructField("season_name", T.StringType())])),
    T.StructField("home_team", T.StructType([
        T.StructField("home_team_id", T.LongType()), T.StructField("home_team_name", T.StringType())])),
    T.StructField("away_team", T.StructType([
        T.StructField("away_team_id", T.LongType()), T.StructField("away_team_name", T.StringType())])),
    T.StructField("home_score", T.IntegerType()),
    T.StructField("away_score", T.IntegerType()),
    T.StructField("competition_stage", named),
    T.StructField("stadium", named),
])

lineup_schema = T.StructType([
    T.StructField("team_id", T.LongType()),
    T.StructField("team_name", T.StringType()),
    T.StructField("lineup", T.ArrayType(T.StructType([
        T.StructField("player_id", T.LongType()),
        T.StructField("player_name", T.StringType()),
        T.StructField("player_nickname", T.StringType()),
        T.StructField("jersey_number", T.IntegerType()),
        T.StructField("country", named),
        T.StructField("positions", T.ArrayType(T.StructType([
            T.StructField("position", T.StringType()),
            T.StructField("from", T.StringType()),
            T.StructField("to", T.StringType()),
            T.StructField("from_period", T.IntegerType()),
            T.StructField("to_period", T.IntegerType()),
        ]))),
    ]))),
])

## 2. Detect new matches (incremental watermark)

In [ ]:
# The partition folders (competition_id=/season_id=) become columns thanks to basePath
bronze_matches = (
    spark.read.schema(match_schema).option("multiLine", True)
    .option("basePath", f"{BRONZE}/matches")
    .json(f"{BRONZE}/matches/competition_id=*/season_id=*/matches.json")
)

if table_exists(f"{SILVER}/matches"):
    processed = spark.read.parquet(f"{SILVER}/matches").select("match_id")
    new_matches = bronze_matches.join(processed, "match_id", "left_anti")
else:
    new_matches = bronze_matches

new_matches = new_matches.cache()
n_new = new_matches.count()
print(f"New matches to process: {n_new}")

In [ ]:
# Build the exact file list for the new matches only (avoids scanning the whole bronze layer)
new_keys = new_matches.select("match_id", "competition_id", "season_id").collect()


def files_for(entity):
    return [f"{BRONZE}/{entity}/competition_id={r.competition_id}/season_id={r.season_id}/{r.match_id}.json"
            for r in new_keys]

## 3. Silver `matches`

In [ ]:
silver_matches = new_matches.select(
    "match_id",
    F.to_date("match_date").alias("match_date"),
    F.col("competition.competition_name").alias("competition_name"),
    F.col("season.season_name").alias("season_name"),
    F.col("competition_stage.name").alias("competition_stage"),
    F.col("home_team.home_team_id").alias("home_team_id"),
    F.col("home_team.home_team_name").alias("home_team"),
    F.col("away_team.away_team_id").alias("away_team_id"),
    F.col("away_team.away_team_name").alias("away_team"),
    "home_score", "away_score",
    F.col("stadium.name").alias("stadium"),
    "competition_id", "season_id",
)

## 4. Silver `events` – flatten the nested JSON
One file = one JSON array of ~3,500 events. `multiLine` reads each array element as a row; the match id
is recovered from the file name and the competition/season from the Hive-style folder names.

In [ ]:
def xy(col, i):
    """Pick the i-th coordinate from a location array."""
    return F.col(col).getItem(i)


if n_new:
    raw_events = (
        spark.read.schema(event_schema).option("multiLine", True)
        .option("basePath", f"{BRONZE}/events")
        .json(files_for("events"))
        .withColumn("match_id", F.regexp_extract(F.input_file_name(), r"(\d+)\.json$", 1).cast("long"))
    )

    events = raw_events.select(
        F.col("id").alias("event_id"), "match_id", "index", "period", "minute", "second",
        F.col("type.name").alias("event_type"),
        F.col("team.id").alias("team_id"), F.col("team.name").alias("team"),
        F.col("player.id").alias("player_id"), F.col("player.name").alias("player"),
        F.col("position.name").alias("position"),
        F.col("play_pattern.name").alias("play_pattern"),
        F.coalesce("under_pressure", F.lit(False)).alias("under_pressure"),
        xy("location", 0).alias("x"), xy("location", 1).alias("y"),
        # End location: whichever nested action carries it
        F.coalesce(xy("pass.end_location", 0), xy("carry.end_location", 0), xy("shot.end_location", 0)).alias("end_x"),
        F.coalesce(xy("pass.end_location", 1), xy("carry.end_location", 1), xy("shot.end_location", 1)).alias("end_y"),
        # Pass details (outcome NULL = completed pass in StatsBomb)
        F.col("pass.recipient.name").alias("pass_recipient"),
        F.col("pass.length").alias("pass_length"),
        F.col("pass.type.name").alias("pass_type"),
        F.col("pass.height.name").alias("pass_height"),
        F.col("pass.outcome.name").alias("pass_outcome"),
        F.coalesce("pass.cross", F.lit(False)).alias("pass_cross"),
        F.coalesce("pass.shot_assist", F.lit(False)).alias("pass_shot_assist"),
        F.coalesce("pass.goal_assist", F.lit(False)).alias("pass_goal_assist"),
        # Shot details
        F.col("shot.statsbomb_xg").alias("shot_xg"),
        F.col("shot.type.name").alias("shot_type"),
        F.col("shot.outcome.name").alias("shot_outcome"),
        F.col("shot.key_pass_id").alias("shot_key_pass_id"),
        # Defensive / ball-progression details
        F.col("dribble.outcome.name").alias("dribble_outcome"),
        F.col("duel.type.name").alias("duel_type"),
        F.col("duel.outcome.name").alias("duel_outcome"),
        F.col("interception.outcome.name").alias("interception_outcome"),
        F.coalesce("ball_recovery.recovery_failure", F.lit(False)).alias("recovery_failure"),
        "competition_id", "season_id",
    ).cache()

## 5. Silver `player_match_minutes` – minutes played from lineups
Lineups store position spells as game-clock strings (`"62:02"`). A `to` of NULL means the player
finished the match, so the spell is closed at the last event timestamp of that match.

In [ ]:
def clock_to_min(c):
    """'MM:SS' -> decimal minutes."""
    parts = F.split(c, ":")
    return parts.getItem(0).cast("double") + parts.getItem(1).cast("double") / 60


if n_new:
    match_end = events.groupBy("match_id").agg(F.max(F.col("minute") + F.col("second") / 60).alias("match_end_min"))

    raw_lineups = (
        spark.read.schema(lineup_schema).option("multiLine", True)
        .option("basePath", f"{BRONZE}/lineups")
        .json(files_for("lineups"))
        .withColumn("match_id", F.regexp_extract(F.input_file_name(), r"(\d+)\.json$", 1).cast("long"))
    )

    # One row per player per match (explode the nested lineup array)
    players = (raw_lineups
               .select("match_id", "competition_id", "season_id", "team_id", "team_name",
                       F.explode("lineup").alias("p"))
               .select("*", "p.*").drop("p"))

    # One row per position spell; unused substitutes have no spells (explode_outer keeps them)
    spells = (players
              .select("match_id", "player_id", F.explode_outer("positions").alias("s"))
              .join(match_end, "match_id")
              .withColumn("start", clock_to_min("s.from"))
              .withColumn("end", F.coalesce(clock_to_min("s.to"), F.col("match_end_min")))
              .withColumn("spell_min", F.greatest(F.col("end") - F.col("start"), F.lit(0.0))))

    # Primary position = position held for the most minutes in the match
    w_pos = Window.partitionBy("match_id", "player_id").orderBy(F.desc("spell_min"))
    primary = (spells.where(F.col("s.position").isNotNull())
               .withColumn("rn", F.row_number().over(w_pos)).where("rn = 1")
               .select("match_id", "player_id", F.col("s.position").alias("primary_position")))

    minutes = spells.groupBy("match_id", "player_id").agg(F.round(F.sum("spell_min"), 2).alias("minutes_played"))

    player_match_minutes = (
        players.select("match_id", "player_id", "player_name", "player_nickname", "jersey_number",
                       F.col("country.name").alias("country"), "team_id", F.col("team_name").alias("team"),
                       "competition_id", "season_id")
        .join(minutes, ["match_id", "player_id"], "left")
        .join(primary, ["match_id", "player_id"], "left")
        .fillna({"minutes_played": 0.0})
    ).cache()

## 6. Data-quality gates (fail fast before anything is written)

In [ ]:
if n_new:
    n_events = events.count()
    dq = {
        "new_matches": n_new,
        "events_rows": n_events,
        "events_null_event_id": events.where(F.col("event_id").isNull()).count(),
        "events_null_match_id": events.where(F.col("match_id").isNull()).count(),
        "events_duplicate_ids": n_events - events.select("event_id").distinct().count(),
        "lineup_rows": player_match_minutes.count(),
        "matches_without_events": n_new - events.select("match_id").distinct().count(),
    }
    assert dq["events_null_event_id"] == 0, "Events without id found"
    assert dq["events_null_match_id"] == 0, "Could not derive match_id from file name"
    assert dq["events_duplicate_ids"] == 0, "Duplicate event ids in batch"
    assert dq["matches_without_events"] == 0, "Some new matches have no event file"
    emit_metrics(dq)

## 7. Write match-level tables (append the new matches)
`matches` is written **last**: it is the watermark, so a failed run never marks matches as processed
without their events.

In [ ]:
if n_new:
    write_table(events, f"{SILVER}/events", ["competition_id", "season_id"], "append")
    write_table(player_match_minutes, f"{SILVER}/player_match_minutes", ["competition_id", "season_id"], "append")
    write_table(silver_matches, f"{SILVER}/matches", ["competition_id", "season_id"], "append")

## 8. Silver `player_season_stats` – recruitment metrics
Recomputed only for the competition-seasons that received new matches. Pitch is 120×80 (StatsBomb),
attacking left ➜ right, goal centre at (120, 40).

* **Progressive pass / carry** – completed, open play, starts beyond x=40 and reduces the distance to goal by ≥25 %.
* **Box entry** – open-play pass or carry that starts outside and ends inside the penalty box (x≥102, 18≤y≤62).
* **xA** – sum of the xG of shots created by the player's key passes.
* **Defensive actions** – won tackles + successful interceptions + ball recoveries + blocks.

In [ ]:
if n_new:
    touched = new_matches.select("competition_id", "season_id").distinct()
    ev = spark.read.parquet(f"{SILVER}/events").join(F.broadcast(touched), ["competition_id", "season_id"])
    mins = spark.read.parquet(f"{SILVER}/player_match_minutes").join(F.broadcast(touched), ["competition_id", "season_id"])

    # Reusable column expressions
    dist_start = F.sqrt((120 - F.col("x")) ** 2 + (40 - F.col("y")) ** 2)
    dist_end = F.sqrt((120 - F.col("end_x")) ** 2 + (40 - F.col("end_y")) ** 2)

    def in_box(cx, cy):
        return (F.col(cx) >= 102) & F.col(cy).between(18, 62)

    set_pieces = ["Corner", "Free Kick", "Throw-in", "Goal Kick", "Kick Off"]
    is_pass = F.col("event_type") == "Pass"
    completed_pass = is_pass & F.col("pass_outcome").isNull()
    open_play_pass = completed_pass & ~F.coalesce(F.col("pass_type").isin(set_pieces), F.lit(False))
    is_carry = F.col("event_type") == "Carry"
    progressive = (F.col("x") >= 40) & (dist_end <= 0.75 * dist_start)
    is_shot = F.col("event_type") == "Shot"
    non_pen_shot = is_shot & (F.coalesce(F.col("shot_type"), F.lit("")) != "Penalty")
    won = ["Won", "Success In Play", "Success Out"]

    def cnt(cond):
        return F.sum(F.when(cond, 1).otherwise(0))

    # xA: attribute each shot's xG back to the passer of its key pass
    xa = (ev.where(is_shot & F.col("shot_key_pass_id").isNotNull())
          .select(F.col("shot_key_pass_id").alias("event_id"), F.col("shot_xg").alias("xa"))
          .join(ev.select("event_id", "player_id", "competition_id", "season_id"), "event_id")
          .groupBy("competition_id", "season_id", "player_id").agg(F.sum("xa").alias("xa")))

    counts = (ev.where(F.col("player_id").isNotNull())
              .groupBy("competition_id", "season_id", "player_id")
              .agg(
                  cnt(non_pen_shot & (F.col("shot_outcome") == "Goal")).alias("np_goals"),
                  F.sum(F.when(non_pen_shot, F.col("shot_xg")).otherwise(0)).alias("np_xg"),
                  cnt(is_shot).alias("shots"),
                  cnt(is_pass).alias("passes_attempted"),
                  cnt(completed_pass).alias("passes_completed"),
                  cnt(F.col("pass_shot_assist") | F.col("pass_goal_assist")).alias("key_passes"),
                  cnt(F.col("pass_goal_assist")).alias("assists"),
                  cnt(open_play_pass & progressive).alias("progressive_passes"),
                  cnt(is_carry & progressive).alias("progressive_carries"),
                  cnt((open_play_pass | is_carry) & in_box("end_x", "end_y") & ~in_box("x", "y")).alias("box_entries"),
                  cnt((F.col("event_type") == "Dribble") & (F.col("dribble_outcome") == "Complete")).alias("successful_dribbles"),
                  cnt((F.col("event_type") == "Dribble") & (F.col("dribble_outcome") == "Complete")
                      & (F.col("x") >= 80)).alias("final_third_dribbles"),
                  cnt(F.col("event_type") == "Pressure").alias("pressures"),
                  cnt((F.col("event_type") == "Pressure") & (F.col("x") >= 80)).alias("final_third_pressures"),
                  cnt(((F.col("duel_type") == "Tackle") & F.col("duel_outcome").isin(won))
                      | ((F.col("event_type") == "Interception") & F.col("interception_outcome").isin(won))
                      | ((F.col("event_type") == "Ball Recovery") & ~F.col("recovery_failure"))
                      | (F.col("event_type") == "Block")).alias("defensive_actions"),
              ))

    # Player attributes: taken from the match where the player played the most minutes
    w_main = Window.partitionBy("competition_id", "season_id", "player_id").orderBy(F.desc("minutes_played"))
    player_attr = (mins.withColumn("rn", F.row_number().over(w_main)).where("rn = 1")
                   .select("competition_id", "season_id", "player_id", "player_name", "player_nickname",
                           "team", "team_id", "country", "jersey_number", "primary_position"))
    played = (mins.groupBy("competition_id", "season_id", "player_id")
              .agg(F.round(F.sum("minutes_played"), 1).alias("minutes"),
                   F.sum(F.when(F.col("minutes_played") > 0, 1).otherwise(0)).alias("appearances")))

    stats = (player_attr.join(played, ["competition_id", "season_id", "player_id"])
             .join(counts, ["competition_id", "season_id", "player_id"], "left")
             .join(xa, ["competition_id", "season_id", "player_id"], "left")
             .fillna(0)
             .where(F.col("minutes") > 0))

    # Position group used for like-for-like comparisons in the gold layer
    pos = F.lower(F.col("primary_position"))
    stats = stats.withColumn(
        "position_group",
        F.when(pos == "goalkeeper", "GK")
         .when(pos.contains("back"), "DEF")
         .when(pos.contains("midfield"), "MID")
         .otherwise("ATT"))

    # Per-90 versions of every counting metric
    per90_cols = ["np_goals", "np_xg", "xa", "shots", "key_passes", "progressive_passes", "progressive_carries",
                  "box_entries", "successful_dribbles", "final_third_dribbles", "pressures",
                  "final_third_pressures", "defensive_actions"]
    for c in per90_cols:
        stats = stats.withColumn(f"{c}_p90", F.round(F.col(c) / F.col("minutes") * 90, 3))
    stats = stats.withColumn(
        "pass_completion_pct",
        F.round(F.when(F.col("passes_attempted") > 0, F.col("passes_completed") / F.col("passes_attempted") * 100), 1))

    write_table(stats, f"{SILVER}/player_season_stats", ["competition_id", "season_id"], "overwrite")
    emit_metrics({"player_season_rows": stats.count(), "touched_partitions": touched.count()})

In [ ]:
# --- Run summary + commit (commit marks the job run as successful for Glue) ---
emit_metrics({"duration_seconds": round(time.time() - RUN_START, 1), "new_matches": n_new})
job.commit()